# ETL y dataset canónico

## tl;dr

El proceso transforma las seis fuentes sin eliminar registros: **220.031 entradas y
220.031 filas canónicas**. Los datos raw permanecen inmutables y cada regla registra las
filas evaluadas y modificadas.

## Contexto y métodos

### Supuestos clave

`reviews_per_month` se conserva como observada. Solo se deriva cero cuando falta y
`number_of_reviews == 0`; si hay reseñas positivas y falta la tasa, la actividad queda
desconocida. No se usan sentinelas de fecha ni ceros para columnas ausentes.

### 1. Cargar el build canónico y su conciliación

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
listings = pd.read_parquet(ROOT / "data/processed/listings.parquet")
transformations = pd.read_parquet(ROOT / "artifacts/quality/transformations.parquet")
reconciliation_path = ROOT / "artifacts/quality/row-reconciliation.json"
reconciliation = json.loads(reconciliation_path.read_text(encoding="utf-8"))
reconciliation

**Conclusión.** El build es aceptable únicamente si la diferencia es cero y la clave
canónica es única. La conciliación demuestra cobertura del proceso, no exactitud del
contenido declarado por cada anuncio.

### 2. Cuantificar tratamientos y disponibilidad analítica

In [ ]:
activity_quality = pd.Series({
    "filas": len(listings),
    "actividad_analizable": int(listings["activity_proxy_is_analyzable"].sum()),
    "ceros_derivados": int(listings["activity_proxy_derived_zero"].sum()),
    "actividad_desconocida": int((~listings["activity_proxy_is_analyzable"]).sum()),
    "precio_invalido": int((~listings["price_is_valid"]).sum()),
}).to_frame("conteo")
display(activity_quality)
transformation_columns = [
    "source_id", "field", "rows_evaluated", "rows_changed", "rule"
]
display(transformations[transformation_columns])

**Conclusión.** Se derivan 54.248 ceros respaldados por ausencia de reseñas y quedan 123
tasas desconocidas pese a existir reseñas positivas. Hay 50 precios no positivos que se
excluyen solo de métricas de precio. Las filas se conservan para el resto del análisis.

### 3. Verificar composición por ciudad y tipología

In [ ]:
city_counts = listings.groupby("city_key", observed=True).size().rename("filas")
room_counts = listings.groupby("room_type", observed=True).size().rename("filas")
display(city_counts.to_frame())
display(room_counts.to_frame())

**Conclusión.** Los tamaños de ciudad son muy distintos, por lo que se usarán cuotas y
comparaciones dentro de ciudad. La tipología mayoritaria global no constituye por sí
sola una oportunidad de captación.

## Takeaways

El dataset canónico está conciliado y conserva indicadores de disponibilidad. Su diseño
permite EDA y estadística sin convertir ausencias en actividad, precio o actualidad.